# RQ1.1 — U-Net на DIVA-HisDB (Colab)

Добивает одностадийную ветку RQ1.1: на U-DIADS-TL сетка закрыта (54/54),
на DIVA посчитано 8 ячеек из 54.

Обучает недостающие **46** ячеек = (энкодер, инициализация, подмножество).
Рецепт совпадает с локальными прогонами, иначе числа несопоставимы:
100 эпох, lr 1e-3, lr бэкбона 1e-4, батч 4, кроп 448, weight decay 1e-4,
patience 30, amp. `k_shot=20` — все размеченные страницы DIVA.

Готовые ячейки пропускаются, прогон возобновляем.

## 1. Конфигурация

In [ ]:

import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = Path("/content/drive/MyDrive/Thesis")
REPO_DIR    = Path("/content/repo")
WORK_DIR    = Path("/content/work")

DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
DRIVE_OUT    = DRIVE_ROOT / "99_evaluation"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"

SUBSETS = ["CB55", "CS18", "CS863"]

# Recipe pinned to the local runs -- changing any of these breaks comparability
# with the 8 cells already in diva_backbone_matrix.csv.
EPOCHS, PATIENCE, LR, LR_BACKBONE = 100, 30, 1e-3, 1e-4
BATCH_SIZE, CROP, WEIGHT_DECAY, SEED = 4, 448, 1e-4, 42
K_SHOT = 20                      # every labelled DIVA page
CONFIG = "configs/unet_resnet_diva.yaml"   # only the encoder is overridden per arm

# (family, bracket, timm encoder, init) -- DINOv3 weights exist only for
# convnext_tiny and vit_small, so those two carry the third init.
GRID = [
    ("convnext", "XS", "tu-convnext_femto",   ["random", "imagenet"]),
    ("convnext", "S",  "tu-convnext_pico",    ["random", "imagenet"]),
    ("convnext", "M",  "tu-convnext_tiny",    ["random", "imagenet", "dinov3"]),
    ("pvt",      "XS", "tu-pvt_v2_b0",        ["random", "imagenet"]),
    ("pvt",      "S",  "tu-pvt_v2_b1",        ["random", "imagenet"]),
    ("pvt",      "M",  "tu-pvt_v2_b2",        ["random", "imagenet"]),
    ("vit",      "XS", "vit_tiny_patch16_224.augreg_in21k",  ["random", "imagenet"]),
    ("vit",      "M",  "vit_small_patch16_224.augreg_in21k", ["random", "imagenet", "dinov3"]),
]
# timm tags that carry the actual weights for each init
WEIGHT_TAG = {
    ("tu-convnext_tiny", "imagenet"): "tu-convnext_tiny.in12k_ft_in1k",
    ("tu-convnext_tiny", "dinov3"):   "tu-convnext_tiny.dinov3_lvd1689m",
    ("vit_small_patch16_224.augreg_in21k", "dinov3"): "vit_small_patch16_224.dino",
}

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def sh(cmd, check=True, cwd=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd)


print(f"ячеек в сетке: {sum(len(i) for *_, i in GRID)} x {len(SUBSETS)} подмножества "
      f"= {sum(len(i) for *_, i in GRID) * len(SUBSETS)}")

## 2. Диск

In [ ]:

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} не найден — создайте его на Диске"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} не найден — залейте туда 00_data"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("диск в порядке:", DRIVE_ROOT)

## 3. Репозиторий

Из клона используются только отслеживаемые файлы: `50_modelling/01_simple_segmentation/`.
Гитигнорные деревья `00_data` и `80_models` подставляются симлинками на локальный SSD —
обучение с FUSE-монтирования Диска работает в разы медленнее.

In [ ]:

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS):
    d.mkdir(parents=True, exist_ok=True)

for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

STAGE_DIR = REPO_DIR / "50_modelling" / "01_simple_segmentation"
assert (STAGE_DIR / "train.py").exists(), "train.py не найден в клоне"
print("репозиторий готов:", REPO_DIR)

## 4. Данные

Копируем с Диска на локальный SSD только то, что нужно одностадийной ветке:
изображения, пиксельная разметка (обучение) и PAGE TASK-2 (оценка).

In [ ]:

NEEDED = []
for s in SUBSETS:
    NEEDED += [
        f"DIVA-HisDB/{s}/img-{s}/img/training",
        f"DIVA-HisDB/{s}/img-{s}/img/validation",
        f"DIVA-HisDB/{s}/img-{s}/img/public-test",
        f"DIVA-HisDB/{s}/pixel-level-gt-{s}/pixel-level-gt/training",
        f"DIVA-HisDB/{s}/pixel-level-gt-{s}/pixel-level-gt/validation",
        f"DIVA-HisDB/{s}/pixel-level-gt-{s}/pixel-level-gt/public-test",
        f"DIVA-HisDB/{s}/PAGE-gt-{s}-TASK-2/TASK-2/validation",
        f"DIVA-HisDB/{s}/PAGE-gt-{s}-TASK-2/TASK-2/public-test",
    ]
JAR_REL = "tools/LineSegmentationEvaluator.jar"

missing = []
for rel in NEEDED:
    src, dst = DRIVE_DATA / rel, LOCAL_DATA / rel
    if dst.is_dir():
        continue
    if not src.is_dir():
        missing.append(rel)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print(f"[copy] {rel} ({len(list(dst.iterdir()))} файлов)")

jar_src, jar_dst = DRIVE_MODELS / JAR_REL, LOCAL_MODELS / JAR_REL
if jar_src.exists() and not jar_dst.exists():
    jar_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(jar_src, jar_dst)
    print("[copy] оценщик DIVA")

assert not missing, f"нет на Диске: {missing[:6]}"
print("данные на месте")

## 5. Зависимости

In [ ]:

sh('pip -q install segmentation-models-pytorch albumentations timm shapely scikit-image')
sh("apt-get -qq install -y default-jre > /dev/null 2>&1 || true", check=False)
sh("nvidia-smi || true", check=False)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} ГБ"
          if hasattr(torch.cuda.get_device_properties(0), "total_mem")
          else f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} ГБ")

## 6. Что уже посчитано

Читает `diva_backbone_matrix.csv` из клона и вычитает готовые ячейки, чтобы
прогон был возобновляемым и не переобучал то, что уже есть локально.

In [ ]:

import pandas as pd

CSV = REPO_DIR / "99_evaluation/01_simple_segmentation/diva_backbone_matrix.csv"


def canon(enc):
    """timm-имя -> (базовый энкодер, инициализация), как в diva_backbone_matrix.csv."""
    e = enc.replace("tu-", "")
    if "dinov3" in e:
        return e.split(".")[0], "dinov3"
    if e in ("dinov2", "dinov2_reg"):
        return e, "ssl"
    if "." in e:
        return e.split(".")[0], "imagenet"
    return e, "random"


done = set()
if CSV.exists():
    for _, r in pd.read_csv(CSV).iterrows():
        base, ini = canon(str(r.encoder))
        done.add((base, ini, str(r.subset)))
print(f"уже посчитано: {len(done)} ячеек")

TODO = []
for fam, bracket, enc, inits in GRID:
    for ini in inits:
        base = enc.replace("tu-", "").split(".")[0]
        for sub in SUBSETS:
            if (base, ini, sub) in done:
                continue
            TODO.append(dict(family=fam, bracket=bracket, encoder=enc, init=ini,
                             subset=sub, base=base))
print(f"к обучению: {len(TODO)} ячеек")
for t in TODO[:5]:
    print("   ", t["family"], t["bracket"], t["base"], t["init"], t["subset"])

## 7. Обучение

Каждая ячейка вызывает репозиторный `train.py` с флагами — конфиг не правится,
поэтому рецепт гарантированно тот же, что локально. Готовая ячейка сразу
уезжает на Диск, так что обрыв сессии не теряет работу.

In [ ]:

def run_name(t):
    return f"unet_{t['encoder']}_diva_{t['subset']}"


def push(name):
    src = LOCAL_MODELS / "01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation" / name
    if src.is_dir():
        dst = DRIVE_MODELS / "01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation" / name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"[push] {name}")


for i, t in enumerate(TODO, 1):
    name = run_name(t)
    out = (LOCAL_MODELS / "01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation"
           / name / "best.pth")
    if out.exists():
        print(f"[skip] ({i}/{len(TODO)}) {name}")
        continue

    enc = WEIGHT_TAG.get((t["encoder"], t["init"]), t["encoder"])
    print(f"\n######## ({i}/{len(TODO)}) {name} | init={t['init']} ########")
    cmd = (
        f"python train.py --config {CONFIG}"
        f" --encoder {enc} --manuscript {t['subset']} --k-shot {K_SHOT}"
        f" --epochs {EPOCHS} --patience {PATIENCE} --lr {LR} --lr-backbone {LR_BACKBONE}"
        f" --batch-size {BATCH_SIZE} --weight-decay {WEIGHT_DECAY} --seed {SEED}"
        f" --run-name {name}"
        f" --out-dir 80_models/01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation"
    )
    r = sh(cmd, check=False, cwd=STAGE_DIR)
    if r.returncode != 0:
        print(f"[FAILED] {name} — пропускаю, остальные продолжают")
        continue
    push(name)

## 8. Оценка

Тот же `evaluate_lines.py`, что и локально, поэтому пять метрик совпадают
по определению с уже посчитанными строками.

In [ ]:

rows = []
for t in TODO:
    name = run_name(t)
    ck = (LOCAL_MODELS / "01_simple_segmentation/diva-hisdb/segmentation/simple_segmentation"
          / name / "best.pth")
    if not ck.exists():
        continue
    cmd = (f"python evaluate_lines.py --family diva --subset {t['subset']}"
           f" --run-name {name} --split public-test")
    r = sh(cmd, check=False, cwd=STAGE_DIR)
    if r.returncode != 0:
        print(f"[FAILED eval] {name}")

for p in sorted((REPO_DIR / "99_evaluation/01_simple_segmentation").glob("*.csv")):
    dst = DRIVE_OUT / "01_simple_segmentation" / p.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst)
    print(f"[push] {p.name}")

## 9. Сводка

После прогона скачайте `99_evaluation/01_simple_segmentation/` с Диска в
локальный репозиторий и пересоберите таблицы:
`python 99_evaluation/make_rq1_split_tables.py`.

In [ ]:

import pandas as pd

if CSV.exists():
    d = pd.read_csv(CSV)
    print(f"строк в матрице: {len(d)}")
    if "subset" in d:
        print(d.groupby("subset").size().to_string())